In [1]:
import pickle
with open('features.pkl', 'rb') as file:
    features = pickle.load(file)
with open('features_ind.pkl', 'rb') as file:
    features_ind = pickle.load(file)
import os
import sys
from utils import CLIP
import numpy as np
import torch
import torch.nn.functional as F
os.chdir('/home/jetson/cmap')
clip = CLIP('ViT-B-16-SigLIP')

/home/jetson/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# text_feature = clip.encode_text(['fire extinguisher'])
text_feature = clip.encode_text(['fire extinguisher'])
similarities = clip.similarity(features, text_feature).squeeze()*100
# get softmax
similarities_s = F.softmax(torch.tensor(similarities), dim=0).numpy()
similarities_s
M_s = np.max(similarities_s)
m_s = np.min(similarities_s)
nsim = np.zeros_like(similarities_s)
nsim[similarities_s > 0.01] = 1
m = int(np.sum(nsim))
# nsim[similarities_s < 0.01] = 0
# n = int(np.sum(nsim))
print(M_s, m_s, m)

import numpy as np
sim_sort_ind = np.argsort(similarities, axis=0)[::-1]
sim_sort_ind = sim_sort_ind[:m]
import matplotlib.pyplot as plt
import cv2
# for i, img_ind in enumerate(sim_sort_ind):
#     plt.figure(i)
#     img = cv2.imread('./athirdmapper/n_images/'+str(img_ind)+'.png')
#     plt.imshow(img)

0.0923719783102673 7.256008069786839e-09 29


In [7]:
# print(sim_sort_ind)
# print(similarities_s[sim_sort_ind])
# get unique points
confidence = {}
for index in sim_sort_ind:
    # print(len(features_ind[index]), features_ind[index])
    similarities_s = similarities[index]
    n_point = len(features_ind[index])
    for point in features_ind[index]:
        confidence[tuple(point)] = confidence.get(tuple(point), 0) + similarities_s/n_point
# sort confidence by value
confidence = dict(sorted(confidence.items(), key=lambda item: item[1], reverse=True))
k = list(confidence.keys())
v = list(confidence.values())
print(k)
print(v)

[(-3.25, 3.0, 0.0), (-3.0, 3.0, 0.0), (-4.25, 2.0, 0.0), (-2.75, 2.75, 0.0), (-3.0, 2.75, 0.0), (-3.0, 2.5, 0.0), (-4.25, 2.25, 0.0), (-3.25, 2.5, 0.0), (-2.5, 2.75, 0.0), (-4.0, 2.0, 0.0), (-4.0, 2.25, 0.0), (-2.75, 2.5, 0.0), (-2.5, 2.5, 0.0), (-2.25, 2.75, 0.0), (-2.0, 2.75, 0.0)]
[39.045712093906104, 36.0264499197721, 9.59030561998778, 7.960623144837038, 6.597709482650771, 6.177641555678243, 5.710421250165609, 4.814727893491977, 4.790433956906695, 3.8798843698221694, 3.8798843698221694, 1.3629136621862667, 1.3629136621862667, 1.3629136621862667, 1.3629136621862667]
